In [41]:
# conda activate genomic_tools

import os
import sys
import pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from collections import defaultdict

sys.path.append("code")

from se_event_utils import *

In [ ]:
# # Load GTF
# gtf = pickle.load(open("data/gencode.v50.annotation_gtf_parsed.pkl", "rb"))

# gtf_exon = gtf[gtf.feature == "exon"]
# exons_by_transcript = {t: grp for t, grp in gtf_exon.groupby('transcript')}  # for transcript lookup
# pickle.dump(exons_by_transcript, open('data/gencode.v46.annotation_exons_by_transcript.pkl', 'wb'))

# gtf_cds = gtf[gtf.feature == "CDS"]
# cds_by_transcript = {t: grp for t, grp in gtf_cds.groupby('transcript')}  # for transcript lookup for coding sequences
# pickle.dump(cds_by_transcript, open('data/gencode.v46.annotation_cds_by_transcript.pkl', 'wb'))

# gtf_transcript = gtf[gtf.feature == "transcript"]
# transcripts_by_gene = {t: grp for t, grp in gtf_transcript.groupby('gene_name')}  # for transcript lookup for coding sequences
# pickle.dump(transcripts_by_gene, open('data/gencode.v46.annotation_transcripts_by_gene.pkl', 'wb'))

In [29]:
exons_by_transcript = pickle.load(open("data/gencode.v46.annotation_exons_by_transcript.pkl", "rb"))
cds_by_transcript = pickle.load(open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb"))
transcripts_by_gene = pickle.load(open("data/gencode.v46.annotation_transcripts_by_gene.pkl", "rb"))

# For each emitted event: catalogue all transcript scenarios 

In [ ]:
# e.g.skipped exon, same exon but different jxns, one shared jxn, etc.
# will need this info interpret functional consequence of splicing

In [ ]:
from multiprocessing import Pool
from functools import partial

In [51]:
# note: this takes about 15 minutes

se_anno_df = pickle.load(open("data/gencode.v50.annotation_SE_anno_df.pkl", "rb"))

event_info = {}
event_dicts = []
for idx, row in se_anno_df.iterrows():
    event = {
        "gene": row['gene_name'],
        "chr": row['chr'],
        "strand": row['strand'],
        "exon_coords": row['exon_coords'],
        "exon_len": row['exon_len'],
        "es": row['exon_start'],
        "ee": row['exon_end'],
        "us_intron_start": row['us_intron_start'],
        "ds_intron_end": row['ds_intron_end']
    } 
    event_info[idx] = annotate_event(event, transcripts_by_gene, exons_by_transcript, cds_by_transcript)
    event_dicts.append({'event': idx, **{k: event[k] for k in ('gene','chr','strand','es','ee')}})

In [52]:
pickle.dump(event_info, open('data/event_info.pkl', 'wb'))

In [53]:
# group events that share junctions
cluster_events(event_dicts) 
pickle.dump(event_dicts, open('data/event_dicts.pkl', 'wb'))


In [ ]:
# add cluster info to `event_info`
for ed in event_dicts:
    event_info[ed['event']]['cluster_id'] = ed['cluster_id']
    
# for each exon_diff_boundary, exon_diff_junction, or skipped variant: flag whether it was observed 
# (or in the case of skipped exons, whether the skipped transcript is compatible with any observed events)
event_coords = {ed['event']: (ed['cluster_id'], ed['es'], ed['ee']) for ed in event_dicts}
event_info = mark_sibling_variants(event_info, event_coords)
pickle.dump(event_info, open('data/event_info.pkl', 'wb'))

In [58]:
event_info['ENSG00000000419_NMD_1']

{'meta': {'gene': 'DPM1',
  'chr': 'chr20',
  'strand': '-',
  'exon_coords': 'chr20:50940865-50940955',
  'exon_len': 91,
  'es': 50940865,
  'ee': 50940955,
  'us_intron_start': 50936263,
  'ds_intron_end': 50942030},
 'cluster_id': 1,
 'compatible': {'ENST00000984927': {'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'nonsense_mediated_decay',
   'transcript_tag': 'TAGENE'},
  'ENST00000466152': {'aa_start': 164,
   'aa_end': 187,
   'coding_nt_length': 70,
   'overlap_type': 'partially_coding',
   'gtf_frame': 1,
   'clean_start': False,
   'clean_end': True,
   'frame_preserving': False,
   'transcript_type': 'nonsense_mediated_decay',
   'transcript_tag': ''},
  'ENST00000684193': {'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'retained_intron',
   'transcript_tag': 'RNA_Seq_supported_only'},
  'ENST00000684708': {'overlap_type': 'noncoding_or_utr',
   'transcript_type': 'retained_intron',
   'transcript_tag': 'RNA_Seq_supported_only'}},
 'exon_diff_junction